# 🤖 Cryptocurrency Trading Bot Demo
## ML/DL/NLP Trading Bot

This notebook demonstrates the key features of the cryptocurrency trading bot.

In [ ]:
# Import required libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (15, 6)

print("Libraries imported successfully!")

## 1. Data Fetching and Technical Indicators

In [ ]:
from crypto_bot.data.fetcher import CryptoDataFetcher

# Initialize data fetcher
fetcher = CryptoDataFetcher()

# Fetch OHLCV data
df = fetcher.fetch_ohlcv('BTC/USDT', '1h', 500)
print(f"Fetched {len(df)} candles")
print("\nFirst few rows:")
df.head()

In [ ]:
# Add technical indicators
df = fetcher.add_technical_indicators(df)
print("Technical indicators added!")
print("\nAvailable columns:")
print(df.columns.tolist())
print("\nLatest data:")
df.tail()

In [ ]:
# Visualize price and indicators
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Price and Moving Averages
axes[0].plot(df.index, df['close'], label='Price', linewidth=2)
axes[0].plot(df.index, df['sma_7'], label='SMA 7', alpha=0.7)
axes[0].plot(df.index, df['sma_25'], label='SMA 25', alpha=0.7)
axes[0].fill_between(df.index, df['bb_lower'], df['bb_upper'], alpha=0.2, label='Bollinger Bands')
axes[0].set_title('Price and Moving Averages', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].set_ylabel('Price (USDT)')

# RSI
axes[1].plot(df.index, df['rsi'], label='RSI', color='purple', linewidth=2)
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].set_title('Relative Strength Index (RSI)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].set_ylabel('RSI')

# MACD
axes[2].plot(df.index, df['macd'], label='MACD', linewidth=2)
axes[2].plot(df.index, df['macd_signal'], label='Signal', linewidth=2)
axes[2].bar(df.index, df['macd_hist'], label='Histogram', alpha=0.3)
axes[2].set_title('MACD', fontsize=14, fontweight='bold')
axes[2].legend()
axes[2].set_ylabel('MACD')

plt.tight_layout()
plt.show()

## 2. Price Prediction with Machine Learning

In [ ]:
from crypto_bot.models.price_predictor import PricePredictionModel

# Initialize model
model = PricePredictionModel(lookback=60, prediction_horizon=1)

# Train model
print("Training model...")
metrics = model.train(df, epochs=50, test_size=0.2)

print("\nTraining Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

In [ ]:
# Make predictions
predictions = model.predict(df.tail(100), steps=10)

print("Next 10 price predictions:")
for i, pred in enumerate(predictions, 1):
    print(f"Step {i}: ${pred:,.2f}")

# Get trading signal
signal = model.get_signal(df.tail(100))
print(f"\nTrading Signal: {signal}")

In [ ]:
# Visualize predictions
recent_prices = df['close'].tail(50).values
timestamps = list(range(len(recent_prices)))
future_timestamps = list(range(len(recent_prices), len(recent_prices) + len(predictions)))

plt.figure(figsize=(15, 6))
plt.plot(timestamps, recent_prices, label='Historical Prices', marker='o', linewidth=2)
plt.plot(future_timestamps, predictions, label='Predictions', marker='s', linewidth=2, color='red', linestyle='--')
plt.axvline(x=len(recent_prices)-1, color='gray', linestyle=':', alpha=0.5)
plt.title('Price Predictions', fontsize=16, fontweight='bold')
plt.xlabel('Time Steps')
plt.ylabel('Price (USDT)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Sentiment Analysis with NLP

In [ ]:
from crypto_bot.models.sentiment_analyzer import SentimentAnalyzer

# Initialize analyzer
analyzer = SentimentAnalyzer()

# Generate sample texts (in production, fetch from Twitter, news, etc.)
texts = analyzer.generate_sample_texts('mixed', 30)

print("Sample texts:")
for i, text in enumerate(texts[:5], 1):
    print(f"{i}. {text}")

In [ ]:
# Analyze sentiment
sentiment_df = analyzer.analyze_batch(texts)

print("Sentiment Analysis Results:")
print(sentiment_df[['text', 'sentiment_label', 'sentiment_score', 'confidence']].head(10))

In [ ]:
# Get overall market sentiment
market_sentiment = analyzer.get_market_sentiment(texts)

print("\nOverall Market Sentiment:")
for key, value in market_sentiment.items():
    if key != 'timestamp':
        print(f"{key}: {value}")

# Get trading signal from sentiment
sentiment_signal = analyzer.get_trading_signal(texts)
print(f"\nSentiment Trading Signal: {sentiment_signal}")

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Sentiment distribution
sentiment_counts = sentiment_df['sentiment_label'].value_counts()
colors = {'positive': 'green', 'negative': 'red', 'neutral': 'gray'}
sentiment_colors = [colors[label] for label in sentiment_counts.index]

axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=sentiment_colors, alpha=0.7)
axes[0].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Sentiment scores
axes[1].hist(sentiment_df['sentiment_score'], bins=20, color='blue', alpha=0.7, edgecolor='black')
axes[1].axvline(x=market_sentiment['avg_sentiment_score'], color='red', linestyle='--', 
                linewidth=2, label=f"Average: {market_sentiment['avg_sentiment_score']:.3f}")
axes[1].set_title('Sentiment Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sentiment Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Complete Trading Bot

In [ ]:
from crypto_bot.bot import CryptoTradingBot

# Initialize bot
bot = CryptoTradingBot(
    symbol='BTC/USDT',
    initial_capital=10000,
    timeframe='1h',
    lookback=60
)

print("Bot initialized successfully!")

In [ ]:
# Train models
print("Training models...\n")
bot.train_models(historical_periods=500)

In [ ]:
# Analyze market
analysis = bot.analyze_market()

print("Market Analysis:")
print(f"Symbol: {analysis['symbol']}")
print(f"Current Price: ${analysis['current_price']:,.2f}")
print(f"\nSignals:")
print(f"  Price Signal: {analysis['price_signal']}")
print(f"  Sentiment Signal: {analysis['sentiment_signal']}")
print(f"  Combined Action: {analysis['combined_signal']['action']}")
print(f"  Confidence: {analysis['combined_signal']['confidence']:.2%}")
print(f"\nMarket Sentiment: {analysis['market_sentiment']['overall_sentiment']}")
print(f"Sentiment Strength: {analysis['market_sentiment']['strength']:.1f}%")

In [ ]:
# Run bot for a few iterations
print("Running trading bot...\n")
final_performance = bot.run(iterations=10, interval=5)

In [ ]:
# Display performance summary
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)

print(f"\nInitial Capital: ${bot.initial_capital:,.2f}")
print(f"Final Capital: ${final_performance['current_capital']:,.2f}")
print(f"Total Return: ${final_performance['total_return']:,.2f} ({final_performance['total_return_pct']:.2f}%)")
print(f"\nTotal Trades: {final_performance['total_trades']}")
print(f"Winning Trades: {final_performance['winning_trades']}")
print(f"Losing Trades: {final_performance['losing_trades']}")
print(f"Win Rate: {final_performance['win_rate']:.2f}%")

if final_performance['total_trades'] > 0:
    print(f"\nAverage Win: ${final_performance['avg_win']:,.2f}")
    print(f"Average Loss: ${final_performance['avg_loss']:,.2f}")

In [ ]:
# Visualize equity curve
if len(bot.strategy.equity_curve) > 0:
    equity_df = pd.DataFrame(bot.strategy.equity_curve)
    
    plt.figure(figsize=(15, 6))
    plt.plot(equity_df.index, equity_df['equity'], linewidth=2, color='blue')
    plt.axhline(y=bot.initial_capital, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
    plt.fill_between(equity_df.index, bot.initial_capital, equity_df['equity'], 
                     where=equity_df['equity'] >= bot.initial_capital, 
                     color='green', alpha=0.3, label='Profit')
    plt.fill_between(equity_df.index, bot.initial_capital, equity_df['equity'], 
                     where=equity_df['equity'] < bot.initial_capital, 
                     color='red', alpha=0.3, label='Loss')
    plt.title('Equity Curve', fontsize=16, fontweight='bold')
    plt.xlabel('Trade Number')
    plt.ylabel('Capital (USDT)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No trades executed to show equity curve")

## 5. Next Steps

### For Development:
1. **Add Real Exchange Connectivity**: Use `ccxt` library to connect to Binance, Coinbase, etc.
2. **Implement Advanced Models**: Use TensorFlow/PyTorch for LSTM/GRU networks
3. **Real Sentiment Data**: Integrate Twitter API, news APIs for live sentiment
4. **Backtesting**: Comprehensive backtesting on historical data
5. **Risk Management**: More sophisticated risk management strategies

### For Deployment:
1. **Web Interface**: Use the Flask web app at `crypto_bot/web/app.py`
2. **Cloud Deployment**: Deploy on AWS, GCP, or Heroku
3. **Database**: Store trades and performance data in PostgreSQL/MongoDB
4. **Monitoring**: Set up alerts and monitoring dashboards
5. **Paper Trading**: Test with paper trading before using real money

### ⚠️ Important:
- Always test thoroughly before using real money
- Understand the risks of cryptocurrency trading
- Use proper risk management
- This is for educational purposes

In [ ]:
print("\n✅ Demo Complete!")
print("\nTo run the web interface:")
print("  cd crypto_bot/web")
print("  python app.py")
print("  Open http://localhost:5000 in your browser")